# Phase 4 — ML Baseline: Random Forest Evaluation & Verification

**Status:** ✅ Completed (pipeline v1.0)  
**Authoritative Script:** `ml_service/src/train_rf.py`  
**Saved Model:** `models/rf_baseline_category_v1.pkl` / `.txt`  
**Metrics Artifact:** `artifacts/rf_metrics_v1.json` & `artifacts/rf_metrics_v1.md`  

---

## Executive Summary
Phase 4 trained the **Random Forest Baseline** using LightGBM's native bagged-tree mode (`boosting_type='rf'`, `bagging_fraction=0.8`, `feature_fraction=0.8`, 100 trees). The model was evaluated against the full un-sampled test set (**504,160 flows**).

### Core Empirical Benchmarks:
- **Binary Intrusion Accuracy:** `98.522%`
- **Attack Detection Rate (Recall / TPR):** `99.940%` (85,097 of 85,148 attacks detected; only 51 missed)
- **False Positive Rate (FPR):** `1.767%` (7,402 benign false alarms out of 419,012 benign flows)
- **Multi-Class Accuracy:** `98.406%` | **Weighted F1:** `0.98988` | **Macro F1:** `0.63921`
- **Operational Latency:** `38.26 μs/sample` | **Training Time:** `140.35 s`

In [ ]:
import json
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import seaborn as sns

root_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from config.paths import Paths

with open(Paths.ARTIFACTS_DIR / 'rf_metrics_v1.json', 'r', encoding='utf-8') as f:
    rf_metrics = json.load(f)

hp = rf_metrics['hyperparameters']
print(f"Model Architecture: Random Forest Baseline (LightGBM boosting_type='rf')")
print(f"Trees / Leaves:    {hp['n_estimators']} trees | {hp['num_leaves']} leaves/tree (max depth: {hp['max_depth']})")
print(f"Test Sample Count: {rf_metrics['test_sample_count']:,}")
print(f"Training Time:     {hp['train_time_seconds']:.2f} s")
print(f"Inference Latency: {rf_metrics['inference_latency_us_per_sample']:.2f} μs/sample")

## 1. Binary Intrusion Confusion Matrix Heatmap
Visualizing Benign vs. Malicious classification boundaries:

In [ ]:
bin_cm = rf_metrics['binary_intrusion_detection']['confusion_matrix']
cm_arr = np.array([
    [bin_cm['true_negatives'], bin_cm['false_positives']],
    [bin_cm['false_negatives'], bin_cm['true_positives']]
])

plt.figure(figsize=(6, 5))
sns.heatmap(cm_arr, annot=True, fmt=',d', cmap='Blues', cbar=False,
            xticklabels=['Pred BENIGN', 'Pred ATTACK'],
            yticklabels=['Actual BENIGN', 'Actual ATTACK'])
plt.title('Random Forest — Binary Intrusion Confusion Matrix', fontsize=12, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

## 2. 15-Class Attack Categorization Performance
Per-class precision, recall, and F1 metrics across all 15 CIC-IDS2017 categories:

In [ ]:
class_rows = []
for name, m in rf_metrics['per_class'].items():
    class_rows.append({
        'Category': name,
        'Test Support': m['support'],
        'Precision': m['precision'],
        'Recall (TPR)': m['recall'],
        'F1-Score': m['f1_score'],
        'Sample Validity': '⚠️ Low Support (<30)' if m['low_support_warning'] else '✅ High Support'
    })

df_rf_classes = pd.DataFrame(class_rows)
df_rf_classes.style.format({
    'Test Support': '{:,}', 'Precision': '{:.4f}',
    'Recall (TPR)': '{:.4f}', 'F1-Score': '{:.4f}'
})

## 3. 15×15 Multi-Class Confusion Matrix Heatmap
Log-transformed heatmap across all 15 threat categories:

In [ ]:
cm_matrix = np.array(rf_metrics['confusion_matrix'])
classes = rf_metrics['classes']

plt.figure(figsize=(12, 10))
# Use log1p to visualize cell densities across extreme sample imbalances
sns.heatmap(np.log1p(cm_matrix), annot=cm_matrix, fmt='d', cmap='YlGnBu', cbar=True,
            xticklabels=classes, yticklabels=classes, annot_kws={'size': 8})
plt.title('Random Forest — 15×15 Multiclass Confusion Matrix (Color=Log1p, Text=Raw Counts)', fontsize=13, fontweight='bold', pad=15)
plt.xlabel('Predicted Category', fontsize=11)
plt.ylabel('Actual Ground Truth Category', fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. Top 20 Feature Importance Attributions
Inspecting which flow features drive the bagged decision trees:

In [ ]:
booster = joblib.load(Paths.model('rf_baseline_category_v1.pkl'))

with open(Paths.SPLITS_DIR / 'split_summary_v1.json', 'r', encoding='utf-8') as f:
    split_meta = json.load(f)
feature_names = split_meta['features']['scaled_feature_names']

importances = booster.feature_importance(importance_type='gain')
df_fi = pd.DataFrame({'Feature': feature_names, 'Gain': importances}).sort_values('Gain', ascending=False).head(20)

plt.figure(figsize=(10, 6))
sns.barplot(data=df_fi, y='Feature', x='Gain', hue='Feature', palette='viridis', legend=False)
plt.title('Random Forest Top 20 Features (Gain)', fontsize=13, fontweight='bold', pad=12)
plt.xlabel('Total Gain Feature Importance', fontsize=11)
plt.tight_layout()
plt.show()

## 5. Interactive Single-Flow Inference Sandbox
Test real flows from the test set through the trained model:

In [ ]:
# Load a single test sample slice
pf_test = pq.ParquetFile(Paths.split('X_test_v1.parquet'))
sample_x = pf_test.read_row_group(0).to_pandas().iloc[50:51].to_numpy(dtype=np.float32)

probs = booster.predict(sample_x)[0]
pred_class_idx = int(np.argmax(probs))
pred_label = classes[pred_class_idx]
confidence = float(probs[pred_class_idx]) * 100

print(f"Predicted Class:      {pred_label}")
print(f"Confidence Score:     {confidence:.2f}%")
print(f"Binary Classification: {'MALICIOUS ATTACK' if pred_class_idx != 0 else 'BENIGN NORMAL'}")

# Top 3 candidate probabilities
top3_indices = np.argsort(probs)[::-1][:3]
print("\nTop Candidate Probabilities:")
for idx in top3_indices:
    print(f"  - {classes[idx]:<25}: {probs[idx]*100:.2f}%")